## defines

In [50]:
from pymongo import MongoClient
from pymongo.operations import SearchIndexModel
from pymongo.errors import OperationFailure
from pprint import pprint
from dotenv import load_dotenv
import os, time
from langchain_core.tools import tool
from typing import List
import voyageai

load_dotenv()
print("MONGO_URI loaded:", bool(os.getenv("MONGO_URI")))
print("GOOGLE_API_KEY loaded:", bool(os.getenv("GOOGLE_API_KEY")))
print("VOYAGEAI_API_KEY loaded:", bool(os.getenv("VOYAGEAI_API_KEY")))

MONGO_URI loaded: True
GOOGLE_API_KEY loaded: True
VOYAGEAI_API_KEY loaded: True


### connect to mongo DB

In [51]:
def init_mongodb():
    """
    Initialize MongoDB client and collections.

    Returns:
        tuple: MongoDB client, vector search collection, full documents collection.
    """
    mongodb_client = MongoClient(os.getenv("MONGO_URI"))
    
    DB_NAME = "ai_agents"
    
    vs_collection = mongodb_client[DB_NAME]["chunked_docs"]
    
    full_collection = mongodb_client[DB_NAME]["full_docs"]
    
    return mongodb_client, vs_collection, full_collection


### generate embedding

In [52]:
def generate_embedding(text: str) -> List[float]:
    """
    Generate embedding for a piece of text.

    Args:
        text (str): The text to embed.
        embedding_model (voyage-4): The embedding model.

    Returns:
        List[float]: The embedding of the text.
    """

    embedding_model = voyageai.Client(api_key=os.getenv("VOYAGEAI_API_KEY"))

    embedding = embedding_model.embed(text, model="voyage-4", input_type="query").embeddings[0]
    
    return embedding


## tool: get_information_for_question_answering

In [53]:
@tool 
def get_information_for_question_answering(user_query: str) -> str:
    """
    Retrieve relevant documents for a user query using vector search.

    Args:
        user_query (str): The user's query.

    Returns:
        str: The retrieved documents as a string.
    """

    query_embedding = generate_embedding(user_query)

    vs_collection = init_mongodb()[1]
    
    pipeline = [
        {
            # Use vector search to find similar documents
            "$vectorSearch": {
                "index": "vector_index",  # Name of the vector index
                "path": "embedding",       # Field containing the embeddings
                "queryVector": query_embedding,  # The query embedding to compare against
                "numCandidates": 150,      # Consider 150 candidates (wider search)
                "limit": 5,                # Return only top 5 matches
            }
        },
        {
            # Project only the fields we need
            "$project": {
                "_id": 0,                  # Exclude document ID
                "body": 1,                 # Include the document body
                "score": {"$meta": "vectorSearchScore"},  # Include the similarity score
            }
        },
    ]
    
    results = vs_collection.aggregate(pipeline)
    
    context = "\n\n".join([doc.get("body") for doc in results])
    
    return context


## tool: get_page_content_for_summarization

In [54]:
@tool 
def get_page_content_for_summarization(user_query: str) -> str:
    """
    Retrieve the content of a documentation page for summarization.

    Args:
        user_query (str): The user's query (title of the documentation page).

    Returns:
        str: The content of the documentation page.
    """
    full_collection = init_mongodb()[2]

    query = {"title": user_query}
    
    projection = {"_id": 0, "body": 1}
    
    document = full_collection.find_one(query, projection)
    
    if document:
        return document["body"]
    else:
        return "Document not found"

## our main function

In [55]:
from langchain_google_genai import ChatGoogleGenerativeAI


def main():
    """
    Main function to initialize and execute the graph.
    """
    # Initialize MongoDB connections
    mongodb_client, vs_collection, full_collection = init_mongodb()
    
    # Initialize the model with API key
    llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.1,
    api_key=os.getenv("GOOGLE_API_KEY")
    )
    
    tools = [
        get_information_for_question_answering,
        get_page_content_for_summarization
    ]

    answer = get_information_for_question_answering.invoke(
    "What are some best practices for data backups in MongoDB?"
    )
    print("answer:" + answer)

    summary = get_page_content_for_summarization.invoke("Create a MongoDB Deployment")
    print("Summary:" + summary)

# invoke main

In [56]:
main()

answer:Create backups using `mongodump` to back up each component in the cluster individually.

Limit the operation of the cluster balancer to provide a window for regular backup operations.

An outline of the procedure and consideration for restoring an *entire* sharded cluster from backup.

# Backup and Restore Sharded Clusters

The following tutorials describe backup and restoration for sharded clusters:

To use `mongodump` and `mongorestore` as a backup strategy for sharded clusters, you must stop the sharded cluster balancer and use the `fsync` command or the `db.fsyncLock()` method on `mongos` to block writes on the cluster during backups.

Sharded clusters can also use one of the following coordinated backup and restore processes, which maintain the atomicity guarantees of transactions across shards:

- MongoDB Atlas

- MongoDB Cloud Manager

- MongoDB Ops Manager

Use file system snapshots back up each component in the sharded cluster individually. The procedure involves stoppi